## Step 1: Read one raw trajectory

The thesis feeds raw sensor trajectories into the model, so first I need to read them correctly. I start with a single product-code file, look at its real structure (delimiter, columns, timestamp format), and confirm what one batch looks like before building anything on top.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE=Path(r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets")
TRAJ=BASE/"Process"

files=sorted(TRAJ.glob("*.csv"))
print("trajectory files found:",len(files))
print("first few:",[f.name for f in files[:5]])

trajectory files found: 25
first few: ['1.csv', '10.csv', '11.csv', '12.csv', '13.csv']


## Look at one file's real structure

Before parsing I check one file directly: how it is separated, what the columns are, and how the first rows look. The dataset is known to use two different formats across files, so I must read what is actually there.

In [2]:
f=TRAJ/"1.csv"

raw=f.read_text().splitlines()[:3]
for i,line in enumerate(raw):
    print("line",i,":",line[:200])

line 0 : timestamp,campaign,batch,code,tbl_speed,fom,main_comp,tbl_fill,SREL,pre_comp,produced,waste,cyl_main,cyl_pre,stiffness,ejection
line 1 : 22-11-2018 23:07,4,26,1,0,0,0.2,4.09,0,0,0,0,1.06,4,3,90
line 2 : 22-11-2018 23:07,4,26,1,0,0,0.2,4.09,0,0,0,0,1.06,4,3,90


## Check the second format

Some files use a different delimiter and timestamp format. I check one of those now, so the loader can detect and handle both correctly instead of assuming one format.

In [3]:
f2=TRAJ/"22.csv"

raw2=f2.read_text().splitlines()[:3]
for i,line in enumerate(raw2):
    print("line",i,":",line[:200])

line 0 : timestamp;campaign;batch;code;tbl_speed;fom;main_comp;tbl_fill;SREL;pre_comp;produced;waste;cyl_main;cyl_pre;stiffness;ejection
line 1 : 2018-12-10 15:10:54;16;70;22;0;0;0;4.24;0;0;0;0;1.37;4.99;0;0
line 2 : 2018-12-10 15:11:04;16;70;22;0;0;0;4.24;0;0;0;0;1.37;4.99;0;0


## A loader that handles both formats

I write one function that reads any trajectory file. It detects the delimiter from the header (comma or semicolon), reads the file, and parses the timestamp with whichever format matches. This way both file types are handled correctly through one path, with no hardcoding per file.

In [4]:
def load_trajectory(path):
    header=path.read_text().splitlines()[0]
    sep=";" if ";" in header else ","
    df=pd.read_csv(path,sep=sep)
    df["timestamp"]=pd.to_datetime(df["timestamp"],format="mixed",dayfirst=True)
    return df

t1=load_trajectory(TRAJ/"1.csv")
t22=load_trajectory(TRAJ/"22.csv")

print("file 1 :",t1.shape,"| first ts:",t1["timestamp"].iloc[0])
print("file 22:",t22.shape,"| first ts:",t22["timestamp"].iloc[0])

file 1 : (106878, 16) | first ts: 2018-11-22 23:07:00
file 22: (329989, 16) | first ts: 2018-12-10 15:10:54


## Split a file into individual batches

Each file is one product code holding many batches stacked together. The model needs one trajectory per batch, so I group the rows by the batch column and check how many batches a file holds and how long each one is.

In [5]:
batches=t1.groupby("batch")
print("file 1 holds",t1["batch"].nunique(),"batches")

lengths=batches.size()
print("rows per batch: min",lengths.min(),"max",lengths.max(),"median",int(lengths.median()))
print()
print("first batch id:",lengths.index[0],"with",lengths.iloc[0],"rows")

file 1 holds 95 batches
rows per batch: min 993 max 1400 median 1104

first batch id: 26 with 1238 rows


## Turn one batch into a sensor array

The model reads only the sensor channels, not the identity columns. I select the ten sensor signals in a fixed order and return them as a numeric array of shape (timesteps, channels) for a single batch. Identity columns (batch, code, campaign) are left out so nothing leaks.

In [6]:
sensors=["tbl_speed","fom","main_comp","tbl_fill","SREL",
         "pre_comp","cyl_main","cyl_pre","stiffness","ejection"]

def batch_to_array(df,batch_id):
    b=df[df["batch"]==batch_id].sort_values("timestamp")
    return b[sensors].to_numpy(dtype="float32")

arr=batch_to_array(t1,26)
print("batch 26 array shape:",arr.shape)
print("channels:",len(sensors))
print("first row:",arr[0])

batch 26 array shape: (1238, 10)
channels: 10
first row: [ 0.    0.    0.2   4.09  0.    0.    1.06  4.    3.   90.  ]


## Step 2: Assemble all batches

I loop over all 25 files, turn every batch into a sensor array, and pair it with its four targets from Laboratory.csv. I keep three aligned lists: the trajectories, the targets, and the product code for each batch (used later for grouped splitting, never as a model input).

In [11]:
lab=pd.read_csv(BASE/"Laboratory.csv",sep=";")
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
lab_idx=lab.set_index("batch")

X_traj=[]
y_list=[]
code_list=[]

for f in files:
    try:
        df=load_trajectory(f)
    except Exception as e:
        print("FAILED",f.name,"->",str(e)[:80])
        continue
    n=0
    for bid in df["batch"].unique():
        if bid in lab_idx.index:
            X_traj.append(batch_to_array(df,bid))
            y_list.append(lab_idx.loc[bid,targets].to_numpy(dtype="float32"))
            code_list.append(int(lab_idx.loc[bid,"code"]))
            n+=1
    print(f.name,"ok,",n,"batches")

y_arr=np.array(y_list,dtype="float32")
groups=np.array(code_list)

print()
print("total batches:",len(X_traj))
print("targets shape:",y_arr.shape)
print("unique codes:",len(np.unique(groups)))

1.csv ok, 95 batches
10.csv ok, 27 batches
11.csv ok, 15 batches
12.csv ok, 21 batches
13.csv ok, 131 batches
14.csv ok, 11 batches
15.csv ok, 64 batches
16.csv ok, 2 batches
17.csv ok, 207 batches
18.csv ok, 1 batches
19.csv ok, 17 batches
2.csv ok, 17 batches
20.csv ok, 4 batches
21.csv ok, 68 batches
22.csv ok, 41 batches
23.csv ok, 187 batches
24.csv ok, 6 batches
25.csv ok, 34 batches
3.csv ok, 16 batches
4.csv ok, 22 batches
5.csv ok, 3 batches
6.csv ok, 2 batches
7.csv ok, 8 batches
8.csv ok, 2 batches
9.csv ok, 4 batches

total batches: 1005
targets shape: (1005, 4)
unique codes: 25


## Save the assembled data

Assembling took a while because it reads all 4.7 million rows. I save the trajectories, targets and group codes to disk so I can reload them instantly later without re-parsing every file.

In [12]:
import pickle

save_path=BASE/"assembled_trajectories.pkl"
with open(save_path,"wb") as fh:
    pickle.dump({"X_traj":X_traj,"y_arr":y_arr,"groups":groups},fh)

print("saved to",save_path)
print("batches:",len(X_traj))

saved to C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets\assembled_trajectories.pkl
batches: 1005


## Step 3: One TCN residual block

The TCN is built from residual blocks. Each block runs two dilated convolutions over the sequence, with a ReLU and dropout between them, then adds the result back to the block input (the residual connection). Dilation lets the block see further back in time without large filters. I stack these blocks with growing dilation so the model captures both short and long patterns in the trajectory.

In [14]:
import torch
import torch.nn as nn

class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None

    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

block=TCNBlock(10,32,dilation=1)
test=torch.randn(2,10,500)
print("block output shape:",block(test).shape)

block output shape: torch.Size([2, 32, 500])


## Step 3b: The full TCN encoder

I stack four residual blocks with dilations 1, 2, 4 and 8, so the model sees increasingly long stretches of the trajectory. After the blocks I average across time, which collapses any length of sequence into one fixed-size summary vector. This is what lets the TCN handle batches of very different lengths, which is the core reason I use it.

In [15]:
class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=64,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,dilation=1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,dilation=2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,dilation=4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,dilation=8,dropout=dropout)

    def forward(self,x):
        x=self.b1(x)
        x=self.b2(x)
        x=self.b3(x)
        x=self.b4(x)
        return x.mean(dim=2)

enc=TCNEncoder()
test=torch.randn(2,10,500)
print("encoder output shape:",enc(test).shape)
n_params=sum(p.numel() for p in enc.parameters())
print("parameters:",n_params)

encoder output shape: torch.Size([2, 64])
parameters: 89152


## Step 3c: Add a prediction head

The encoder gives a 64-number summary of each trajectory. I add a small feed-forward head that maps this summary to one predicted value. For this first model I predict dissolution, the target with the most room to improve over the baselines. Multi-task heads for all four targets come later.

In [16]:
class TrajModel(nn.Module):
    def __init__(self,in_ch=10,hidden=64,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.head=nn.Sequential(
            nn.Linear(hidden,32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32,1)
        )

    def forward(self,x):
        z=self.encoder(x)
        return self.head(z).squeeze(-1)

model=TrajModel()
test=torch.randn(2,10,500)
print("model output shape:",model(test).shape)
print("total parameters:",sum(p.numel() for p in model.parameters()))

model output shape: torch.Size([2])
total parameters: 91265


## Step 4: Prepare data for training

To train on batches of different lengths, I pad each batch of trajectories to the same length and normalise the ten sensor channels. Normalisation uses statistics from the training set only, so no information leaks from the test set. For this first model I cap very long trajectories at a fixed length to keep CPU training feasible, and I note this so it can be lifted later on GPU.

In [18]:
STRIDE=3

def prep_batch(traj_list,mean,std,maxlen):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

ds_lengths=[len(a[::STRIDE]) for a in X_traj]
MAXLEN=int(np.percentile(ds_lengths,95))
print("after stride",STRIDE,": median",int(np.median(ds_lengths)),"95th pct",MAXLEN,"max",max(ds_lengths))

after stride 3 : median 1076 95th pct 6467 max 13904


In [19]:
MAXLEN=2000
print("using MAXLEN",MAXLEN,"after stride",STRIDE)
covered=sum(1 for a in X_traj if len(a[::STRIDE])<=MAXLEN)
print("batches fully covered:",covered,"of",len(X_traj),f"({100*covered/len(X_traj):.0f}%)")

using MAXLEN 2000 after stride 3
batches fully covered: 798 of 1005 (79%)


## Step 4b: Train and evaluate with grouped CV

I train the model with GroupKFold on product code, the same honest setup as the baselines, so no product is in both training and test. For each fold I compute normalisation from the training batches only, train for a set number of epochs, and measure RMSE on the held-out products. This keeps the deep model directly comparable to the baselines.

In [23]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

device="cpu"
target_idx=0

def train_eval(n_splits=3,epochs=8,lr=5e-4,batch_size=32):
    gkf=GroupKFold(n_splits=n_splits)
    yt=y_arr[:,target_idx]
    fold_rmse=[]
    for fold,(tr,te) in enumerate(gkf.split(X_traj,yt,groups=groups)):
        mean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
        std=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
        Xtr=prep_batch([X_traj[i] for i in tr],mean,std,MAXLEN).to(device)
        Xte=prep_batch([X_traj[i] for i in te],mean,std,MAXLEN).to(device)
        ytr=torch.tensor(yt[tr]).to(device)
        yte=yt[te]
        model=TrajModel().to(device)
        opt=torch.optim.Adam(model.parameters(),lr=lr)
        lossf=nn.MSELoss()
        model.train()
        for ep in range(epochs):
            perm=torch.randperm(len(Xtr))
            for i in range(0,len(Xtr),batch_size):
                idx=perm[i:i+batch_size]
                opt.zero_grad()
                pred=model(Xtr[idx])
                loss=lossf(pred,ytr[idx])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step()
        model.eval()
        with torch.no_grad():
            pred=model(Xte).cpu().numpy()
        if np.isnan(pred).any():
            print(f"fold {fold+1}: predictions contain NaN, skipping")
            continue
        rmse=np.sqrt(mean_squared_error(yte,pred))
        fold_rmse.append(rmse)
        print(f"fold {fold+1}: RMSE {rmse:.3f}")
    if fold_rmse:
        print("mean RMSE:",round(np.mean(fold_rmse),3))
    return fold_rmse

print("ready to train (with gradient clipping)")

ready to train (with gradient clipping)


## Step 4c: Run the training

This trains the first model and prints the RMSE for each fold, then the average. It is the slowest cell because it trains a real network on CPU across the folds. The result is directly comparable to the dissolution baseline.

In [24]:
import time
t0=time.time()
results=train_eval(n_splits=3,epochs=8)
print("time taken:",round(time.time()-t0),"seconds")

fold 1: predictions contain NaN, skipping
fold 2: predictions contain NaN, skipping
fold 3: predictions contain NaN, skipping
time taken: 720 seconds


In [25]:
nan_batches=0
for i,a in enumerate(X_traj):
    if np.isnan(a).any():
        nan_batches+=1
inf_batches=sum(1 for a in X_traj if np.isinf(a).any())
print("batches with NaN:",nan_batches)
print("batches with Inf:",inf_batches)

allvals=np.vstack([a[::STRIDE] for a in X_traj])
print("global min:",np.nanmin(allvals),"max:",np.nanmax(allvals))
print("any NaN in stacked:",np.isnan(allvals).any())

batches with NaN: 62
batches with Inf: 0
global min: 0.0 max: 4747.0
any NaN in stacked: True


In [26]:
cleaned=0
for i in range(len(X_traj)):
    if np.isnan(X_traj[i]).any():
        X_traj[i]=np.nan_to_num(X_traj[i],nan=0.0).astype("float32")
        cleaned+=1

print("cleaned batches:",cleaned)
print("any NaN left:",any(np.isnan(a).any() for a in X_traj))

cleaned batches: 62
any NaN left: False


In [27]:
import time
t0=time.time()
results=train_eval(n_splits=2,epochs=3)
print("time taken:",round(time.time()-t0),"seconds")

fold 1: RMSE 44.209
fold 2: RMSE 28.589
mean RMSE: 36.399
time taken: 141 seconds


In [28]:
print("dissolution range:",y_arr[:,0].min(),"to",y_arr[:,0].max())
print("mean:",round(float(y_arr[:,0].mean()),2),"std:",round(float(y_arr[:,0].std()),2))

dissolution range: 82.5 to 102.67
mean: 90.65 std: 3.36


In [29]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error

device="cpu"
target_idx=0

def train_eval(n_splits=3,epochs=8,lr=5e-4,batch_size=32):
    gkf=GroupKFold(n_splits=n_splits)
    yt=y_arr[:,target_idx]
    fold_rmse=[]
    for fold,(tr,te) in enumerate(gkf.split(X_traj,yt,groups=groups)):
        mean=np.vstack([X_traj[i][::STRIDE] for i in tr]).mean(axis=0)
        std=np.vstack([X_traj[i][::STRIDE] for i in tr]).std(axis=0)+1e-6
        Xtr=prep_batch([X_traj[i] for i in tr],mean,std,MAXLEN).to(device)
        Xte=prep_batch([X_traj[i] for i in te],mean,std,MAXLEN).to(device)
        ymean=yt[tr].mean()
        ystd=yt[tr].std()+1e-6
        ytr=torch.tensor((yt[tr]-ymean)/ystd).to(device)
        yte=yt[te]
        model=TrajModel().to(device)
        opt=torch.optim.Adam(model.parameters(),lr=lr)
        lossf=nn.MSELoss()
        model.train()
        for ep in range(epochs):
            perm=torch.randperm(len(Xtr))
            for i in range(0,len(Xtr),batch_size):
                idx=perm[i:i+batch_size]
                opt.zero_grad()
                pred=model(Xtr[idx])
                loss=lossf(pred,ytr[idx])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step()
        model.eval()
        with torch.no_grad():
            pred=model(Xte).cpu().numpy()*ystd+ymean
        if np.isnan(pred).any():
            print(f"fold {fold+1}: NaN, skipping")
            continue
        rmse=np.sqrt(mean_squared_error(yte,pred))
        fold_rmse.append(rmse)
        print(f"fold {fold+1}: RMSE {rmse:.3f}")
    if fold_rmse:
        print("mean RMSE:",round(np.mean(fold_rmse),3))
    return fold_rmse

print("ready (with target scaling)")

ready (with target scaling)


In [30]:
import time
t0=time.time()
results=train_eval(n_splits=2,epochs=3)
print("time taken:",round(time.time()-t0),"seconds")

fold 1: RMSE 3.398
fold 2: RMSE 3.422
mean RMSE: 3.41
time taken: 131 seconds


In [31]:
import time
t0=time.time()
results=train_eval(n_splits=3,epochs=10)
print("time taken:",round(time.time()-t0),"seconds")

fold 1: RMSE 2.905
fold 2: RMSE 2.961
fold 3: RMSE 4.221
mean RMSE: 3.362
time taken: 892 seconds


In [32]:
import json
result_summary={
"target":"dissolution_av",
"model":"single-task TCN (first version)",
"setup":"3-fold GroupKFold on code, 10 epochs, CPU, stride-3 downsample, MAXLEN 2000",
"fold_rmse":[round(float(r),3) for r in results],
"mean_rmse":round(float(np.mean(results)),3),
"baseline_lasso":3.119,
"baseline_xgb":3.147,
"baseline_dummy":3.512
}
with open(BASE/"first_model_result.json","w") as fh:
    json.dump(result_summary,fh,indent=2)
print(json.dumps(result_summary,indent=2))

{
  "target": "dissolution_av",
  "model": "single-task TCN (first version)",
  "setup": "3-fold GroupKFold on code, 10 epochs, CPU, stride-3 downsample, MAXLEN 2000",
  "fold_rmse": [
    2.905,
    2.961,
    4.221
  ],
  "mean_rmse": 3.362,
  "baseline_lasso": 3.119,
  "baseline_xgb": 3.147,
  "baseline_dummy": 3.512
}


In [2]:
import pickle
import numpy as np
import pandas as pd

BASE_path=r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets"
with open(BASE_path+r"\assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)

X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

print("loaded:",len(X_traj),"batches")

loaded: 1005 batches


In [3]:
import pandas as pd
import numpy as np

stride=3
maxlen=2000

rows=[]
for i in range(len(X_traj)):
    full_len=len(X_traj[i])
    ds_len=len(X_traj[i][::stride])
    truncated=ds_len>maxlen
    rows.append({"code":int(groups[i]),"truncated":truncated})

chk=pd.DataFrame(rows)

print("=== TRUNCATED BATCHES (21%) BY PRODUCT CODE ===")
trunc_by_code=chk.groupby("code")["truncated"].agg(["sum","count"])
trunc_by_code["pct"]=(100*trunc_by_code["sum"]/trunc_by_code["count"]).round(0)
print(trunc_by_code[trunc_by_code["sum"]>0].sort_values("sum",ascending=False))

=== TRUNCATED BATCHES (21%) BY PRODUCT CODE ===
      sum  count    pct
code                   
13     86    131   66.0
22     33     41   80.0
12     18     21   86.0
2      16     17   94.0
15     13     64   20.0
14     11     11  100.0
17     10    207    5.0
23      6    187    3.0
5       3      3  100.0
6       2      2  100.0
16      2      2  100.0
8       2      2  100.0
7       1      8   12.0
10      1     27    4.0
11      1     15    7.0
18      1      1  100.0
24      1      6   17.0


In [4]:
print("load_trajectory defined:", "load_trajectory" in dir())
print("batch_to_array defined:", "batch_to_array" in dir())
print("files defined:", "files" in dir())

load_trajectory defined: False
batch_to_array defined: False
files defined: False


## Reload the reader functions

The kernel restarted, so I redefine the trajectory loader and the batch extractor, and rebuild the file list. This is just function definitions, no heavy processing.

In [5]:
from pathlib import Path

BASE=Path(r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets")
TRAJ=BASE/"Process"
files=sorted(TRAJ.glob("*.csv"))

sensors=["tbl_speed","fom","main_comp","tbl_fill","SREL",
         "pre_comp","cyl_main","cyl_pre","stiffness","ejection"]

def load_trajectory(path):
    header=path.read_text().splitlines()[0]
    sep=";" if ";" in header else ","
    df=pd.read_csv(path,sep=sep)
    ts=df["timestamp"].astype(str)
    for fmt in ["%d-%m-%Y %H:%M","%Y-%m-%d %H:%M:%S","%d%m%Y %H:%M","%d-%m-%Y %H:%M:%S","%Y-%m-%d %H:%M"]:
        try:
            df["timestamp"]=pd.to_datetime(ts,format=fmt)
            return df
        except ValueError:
            continue
    raise ValueError(f"unknown timestamp format in {path.name}")

def batch_to_array(df,batch_id):
    b=df[df["batch"]==batch_id].sort_values("timestamp")
    return b[sensors].to_numpy(dtype="float32")

print("functions ready, files found:",len(files))

functions ready, files found: 25


## Do the NaN-sensor batches cluster by product code?

The mentor asked whether the 62 batches that had missing sensor readings cluster around particular product codes. If they do, it interacts with the leave-one-product-code-out design and needs noting in the methodology. I re-read the raw files, flag any batch containing NaN before cleaning, and group the count by product code.

In [6]:
nan_rows=[]
for f in files:
    df=load_trajectory(f)
    for bid in df["batch"].unique():
        arr=batch_to_array(df,bid)
        nan_rows.append({"code":int(df[df["batch"]==bid]["code"].iloc[0]),
                         "has_nan":bool(np.isnan(arr).any())})

nan_chk=pd.DataFrame(nan_rows)
print("total batches with NaN:",int(nan_chk["has_nan"].sum()))
print()
nan_by_code=nan_chk.groupby("code")["has_nan"].agg(["sum","count"])
nan_by_code["pct"]=(100*nan_by_code["sum"]/nan_by_code["count"]).round(0)
print(nan_by_code[nan_by_code["sum"]>0].sort_values("sum",ascending=False))

total batches with NaN: 62

      sum  count   pct
code                  
17     13    207   6.0
13     11    131   8.0
23     10    187   5.0
15      8     64  12.0
10      5     27  19.0
22      4     41  10.0
14      3     11  27.0
12      2     21  10.0
1       1     95   1.0
4       1     22   5.0
11      1     15   7.0
5       1      3  33.0
21      1     68   1.0
25      1     34   3.0
